Авторизация в kaggle

In [ ]:
import kagglehub
kagglehub.login()

Загрузка данных

In [ ]:
ml_intensive_yandex_academy_spring_2026_path = kagglehub.competition_download('ml-intensive-yandex-academy-spring-2026')

print('Data source import complete.')

In [ ]:
!pip install torchmetrics

import os
import warnings
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.v2 as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from IPython.display import clear_output
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch.optim import Optimizer
from sklearn.model_selection import train_test_split
from torchmetrics.classification import BinaryF1Score
from sklearn.metrics import f1_score

CUDA vs CPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Отключение warning'ов для более чистого вывода

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
# пути к файлам если запускать в блокноте kaggle'а
# path_train_images = "/kaggle/input/competitions/ml-intensive-yandex-academy-spring-2026/dataset/train_images"
# path_test_images = "/kaggle/input/competitions/ml-intensive-yandex-academy-spring-2026/dataset/test_images"
# path_train_solution_csv = "/kaggle/input/competitions/ml-intensive-yandex-academy-spring-2026/dataset/train_solution.csv"

# пути к файлам если запускать в колабе
path_train_images = "/root/.cache/kagglehub/competitions/ml-intensive-yandex-academy-spring-2026/dataset/train_images"
path_test_images = "/root/.cache/kagglehub/competitions/ml-intensive-yandex-academy-spring-2026/dataset/test_images"
path_train_solution_csv = "/root/.cache/kagglehub/competitions/ml-intensive-yandex-academy-spring-2026/dataset/train_solution.csv"

In [ ]:
# нормализация одного residual-канала в диапазон [0, 1]
def normalize_channel(channel):
    channel = channel.astype(np.float32)

    # мин. и макс. значение в канале
    channel_min = channel.min()
    channel_max = channel.max()

    # если канал не вырожденный, нормализуем его в [0, 1]
    if channel_max - channel_min > 1e-6:
        channel = (channel - channel_min) / (channel_max - channel_min)
    else:
        channel = np.zeros_like(channel, dtype=np.float32)

    return channel


# построение multi-channel residual представления: high-pass, Laplacian, Sobel X, Sobel Y
def make_multichannel_residual_image(image):
    # перевод RGB-изображения в оттенки серого
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # строим high-pass компоненту: вычитаем размытую версию изображения из исходной
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    high_pass = gray.astype(np.float32) - blurred.astype(np.float32)

    # Laplacian подчёркивает резкие локальные изменения яркости
    laplacian = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)

    # Sobel X выделяет вертикальные границы
    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)

    # Sobel Y выделяет горизонтальные границы
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    # нормализация каналов отдельно
    high_pass = normalize_channel(high_pass)
    laplacian = normalize_channel(laplacian)
    sobel_x = normalize_channel(sobel_x)
    sobel_y = normalize_channel(sobel_y)

    # склеиваем 4 residual-канала в один многоканальный массив
    residual = np.stack(
        [high_pass, laplacian, sobel_x, sobel_y],
        axis=-1
    )

    return residual

Специальные классы Dataset

In [ ]:
# Dataset для обучения и валидации
# возврат:
# - rgb_image: тензор с RGB-изображением
# - residual_image: тензор с residual-признаками
# - label: метка класса (0 или 1)
class FaceTrainDualDataset(Dataset):
    def __init__(self, images_folder, labels_csv, rgb_transform=None, residual_transform=None, indices=None):
        self.images_folder = images_folder
        self.rgb_transform = rgb_transform
        self.residual_transform = residual_transform

        df = pd.read_csv(labels_csv, header=None, names=["Id", "target_feature"])

        # если переданы индексы (train/valid split), берём только нужные строки
        if indices is not None:
            df = df.iloc[indices].reset_index(drop=True)

        self.ids = df["Id"].astype(int).tolist()
        self.labels = df["target_feature"].astype(int).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        label = self.labels[index]

        # загрузка изображений
        image_path = os.path.join(self.images_folder, f"{image_id}.jpg")
        image = Image.open(image_path).convert("RGB")
        image = np.array(image)

        # строим residual-признаки
        residual = make_multichannel_residual_image(image)

        # Применение аугментаций отдельно к RGB и residual
        if self.rgb_transform is not None:
            rgb_image = self.rgb_transform(image=image)["image"]
        else:
            rgb_image = image

        if self.residual_transform is not None:
            residual_image = self.residual_transform(image=residual)["image"]
        else:
            residual_image = residual

        return rgb_image, residual_image, label


# Dataset для тестовой выборки
# возврат:
# - RGB-изображение
# - residual-представление
# - изображения
class FaceTestDualDataset(Dataset):
    def __init__(self, images_folder, rgb_transform=None, residual_transform=None):
        self.images_folder = images_folder
        self.rgb_transform = rgb_transform
        self.residual_transform = residual_transform

        self.image_files = sorted(
            [filename for filename in os.listdir(images_folder) if filename.lower().endswith(".jpg")],
            key=lambda filename: int(os.path.splitext(filename)[0])
        )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        filename = self.image_files[index]
        image_id = int(os.path.splitext(filename)[0])

        image_path = os.path.join(self.images_folder, filename)
        image = Image.open(image_path).convert("RGB")
        image = np.array(image)

        residual = make_multichannel_residual_image(image)

        if self.rgb_transform is not None:
            rgb_image = self.rgb_transform(image=image)["image"]
        else:
            rgb_image = image

        if self.residual_transform is not None:
            residual_image = self.residual_transform(image=residual)["image"]
        else:
            residual_image = residual

        return rgb_image, residual_image, image_id

Аугментации

In [ ]:
train_rgb_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.MedianBlur(blur_limit=3, p=0.2),
    A.Normalize(mean=(0.5192, 0.4276, 0.3844), std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

test_rgb_transforms = A.Compose([
    A.Normalize(mean=(0.5192, 0.4276, 0.38445), std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

train_residual_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.5192, 0.4276, 0.3844), std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

test_residual_transforms = A.Compose([
    A.Normalize(mean=(0.5192, 0.4276, 0.3844), std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

Методы для создания loader'ов

In [ ]:
# создание train и validation loader'ов для двухветочной модели
def make_train_valid_dual_loaders(
    images_folder,
    labels_csv,
    train_rgb_transform=None,
    valid_rgb_transform=None,
    train_residual_transform=None,
    valid_residual_transform=None,
    batch_size=32,
    valid_size=0.2,
    random_state=42,
    shuffle_train=True,
    num_workers=0
):
    # загрузка таблицы с метками классов
    df = pd.read_csv(labels_csv, header=None, names=["Id", "target_feature"])
    all_indices = np.arange(len(df))

    # деление индексов на train и valid с сохранением пропорций классов
    train_indices, valid_indices = train_test_split(
        all_indices,
        test_size=valid_size,
        random_state=random_state,
        stratify=df["target_feature"]
    )

    # Dataset для обучения
    train_dataset = FaceTrainDualDataset(
        images_folder=images_folder,
        labels_csv=labels_csv,
        rgb_transform=train_rgb_transform,
        residual_transform=train_residual_transform,
        indices=train_indices
    )

    # Dataset для валидации
    valid_dataset = FaceTrainDualDataset(
        images_folder=images_folder,
        labels_csv=labels_csv,
        rgb_transform=valid_rgb_transform,
        residual_transform=valid_residual_transform,
        indices=valid_indices
    )

    # DataLoader для обучения
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        pin_memory=True
    )

    # DataLoader для валидации
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, valid_loader


# создание test loader для двухветочной модели,
# для теста меток нет, поэтому Dataset возвращает:
# - RGB изображение
# - residual изображение
# - id изображения
def make_test_dual_loader(
    images_folder,
    rgb_transform=None,
    residual_transform=None,
    batch_size=128,
    shuffle=False,
    num_workers=0
):
    test_dataset = FaceTestDualDataset(
        images_folder=images_folder,
        rgb_transform=rgb_transform,
        residual_transform=residual_transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True
    )

    return test_loader

In [ ]:
# небольшой класс для сжатия размера кода далее
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)


# двухветочная модель:
# 1) RGB-ветка - учится на исходном изображении
# 2) Residual-ветка - учится на low-level признаках (артефакты)
# на выходе признаки объединяются и подаются в классификатор
class DualBranchResidualNet(nn.Module):
    def __init__(self):
        super().__init__()

        # ветка для RGB-изображения
        self.rgb_branch = nn.Sequential(
            ConvBlock(3, 32),
            nn.MaxPool2d(2),

            ConvBlock(32, 64),
            nn.MaxPool2d(2),

            ConvBlock(64, 128),
            nn.MaxPool2d(2),

            ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # ветка для residual-признаков (4 канала)
        self.residual_branch = nn.Sequential(
            ConvBlock(4, 16),
            nn.MaxPool2d(2),

            ConvBlock(16, 32),
            nn.MaxPool2d(2),

            ConvBlock(32, 64),
            nn.MaxPool2d(2),

            ConvBlock(64, 128),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # общий классификатор
        self.head = nn.Sequential(
            nn.Linear(256 + 128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, rgb_image, residual_image):
        # извлекаем признаки из обеих веток
        rgb_features = self.rgb_branch(rgb_image)
        residual_features = self.residual_branch(residual_image)

        # преобразуем в векторы
        rgb_features = rgb_features.flatten(1)
        residual_features = residual_features.flatten(1)

        # объединяем признаки
        features = torch.cat([rgb_features, residual_features], dim=1)

        # предсказание
        logits = self.head(features)

        return logits

Создание loader'ов

In [ ]:
train_loader, valid_loader = make_train_valid_dual_loaders(
    images_folder=path_train_images,
    labels_csv=path_train_solution_csv,
    train_rgb_transform=train_rgb_transforms,
    valid_rgb_transform=test_rgb_transforms,
    train_residual_transform=train_residual_transforms,
    valid_residual_transform=test_residual_transforms,
    batch_size=64,
    valid_size=0.2,
    random_state=100,
    shuffle_train=True,
    num_workers=4
)

test_loader = make_test_dual_loader(
    images_folder=path_test_images,
    rgb_transform=test_rgb_transforms,
    residual_transform=test_residual_transforms,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

In [ ]:
# обучение на одной эпохе
def train(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: Optimizer,
    loss_fn,
    device: torch.device
):
    model.train()

    total_loss = 0.0

    for rgb_x, residual_x, y in tqdm(data_loader):
        rgb_x = rgb_x.to(device)
        residual_x = residual_x.to(device)
        y = y.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        output = model(rgb_x, residual_x)
        loss = loss_fn(output, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


# оценка модели
@torch.inference_mode()
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    loss_fn,
    device: torch.device
):
    model.eval()

    total_loss = 0.0
    f1_metric = BinaryF1Score().to(device)

    for rgb_x, residual_x, y in tqdm(data_loader):
        rgb_x = rgb_x.to(device)
        residual_x = residual_x.to(device)
        y = y.to(device).float().unsqueeze(1)

        output = model(rgb_x, residual_x)
        loss = loss_fn(output, y)

        total_loss += loss.item()

        # перевод logits в вероятности
        probs = torch.sigmoid(output).squeeze(1)
        targets = y.squeeze(1).int()

        # обновление метрики
        f1_metric.update(probs, targets)

    return total_loss / len(data_loader), f1_metric.compute().item()


# построение графиков
def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    valid_f1: list[float],
    title: str
):
    plt.figure(figsize=(16, 8))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + ' f1')

    plt.plot(valid_f1, label='Valid f1')
    plt.legend()

    plt.show()


# полный цикл обучения
def fit(model, train_loader, valid_loader, optimizer, loss_fn, device, num_epochs, title):
    train_loss_history, valid_loss_history = [], []
    valid_f1_history = []

    best_valid_f1 = -1.0

    for epoch in range(num_epochs):
        train_loss = train(model, train_loader, optimizer, loss_fn, device)
        valid_loss, valid_f1 = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)
        valid_f1_history.append(valid_f1)

        clear_output()

        plot_stats(
            train_loss_history,
            valid_loss_history,
            valid_f1_history,
            title
        )

        print(
            f"epoch={epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"valid_loss={valid_loss:.4f} | "
            f"valid_f1={valid_f1:.4f}"
        )

        # сохранение лучшей модели
        if valid_f1 > best_valid_f1:
            best_valid_f1 = valid_f1
            torch.save(model.state_dict(), "best_model.pt")
            torch.save(optimizer.state_dict(), "best_optimizer.pt")
            print(f"best model epoch {epoch}")

    history = {
        "train_loss": train_loss_history,
        "valid_loss": valid_loss_history,
        "valid_f1": valid_f1_history
    }

    return history


# создание файла с сопоставлением номера изображения к предсказанию
@torch.inference_mode()
def predict_test(model, test_loader, device, threshold=0.45):
    model.eval()

    all_ids = []
    all_preds = []

    for rgb_x, residual_x, ids in tqdm(test_loader):
        rgb_x = rgb_x.to(device)
        residual_x = residual_x.to(device)

        output = model(rgb_x, residual_x)
        probs = torch.sigmoid(output).squeeze(1)
        preds = (probs >= threshold).long()

        all_ids.extend(ids.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

    submission = pd.DataFrame({
        "id": all_ids,
        "target_feature": all_preds
    })

    submission = submission.sort_values("id").reset_index(drop=True)
    submission.to_csv("submission_multires.csv", index=False)

    return submission

Подсчёт веса положительного класса для BCEWithLogitsLoss. Это нужно для компенсации дисбаланса классов: ошибки на более редком положительном классе будут штрафоваться сильнее.

In [ ]:
df = pd.read_csv(path_train_solution_csv, header=None, names=["id", "target_feature"])

# подсчёт количества объектов положительного и отрицательного класса
num_pos = (df["target_feature"] == 1).sum()
num_neg = (df["target_feature"] == 0).sum()

# вес положительного класса: чем реже положительный класс, тем больше этот коэффициент
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

print("num_pos =", num_pos)
print("num_neg =", num_neg)
print("pos_weight =", pos_weight.item())

In [ ]:
model = DualBranchResidualNet().to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# запуск обучения
history = fit(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    loss_fn=criterion,
    device=device,
    num_epochs=10,
    title="multi_residual"
)

In [ ]:
# генерация предсказаний для теста
submission = predict_test(
    model=model,
    test_loader=test_loader,
    device=device,
    threshold=0.45,
)

submission.head()

Загрузка весов обученной модели

In [ ]:
weights_path = "best_model.pt"

model = DualBranchResidualNet().to(device)
model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()

print("Model weights loaded")

In [ ]:
valid_loss, valid_f1 = evaluate(
    model=model,
    data_loader=valid_loader,
    loss_fn=criterion,
    device=device
)

print("Validation loss:", valid_loss)
print("Validation F1:", valid_f1)

# Описание решения

В данной работе исследовались несколько подходов к детекции дипфейков.
В качестве финального решения была выбрана двухветочная сверточная сеть.

## Идея модели
Модель состоит из двух веток:
1. RGB-ветка - обрабатывает исходное цветное изображение.
2. Residual-ветка - обрабатывает набор low-level признаков, построенных из изображения:
   - high-pass,
   - Laplacian,
   - Sobel X,
   - Sobel Y.

Признаки из двух веток объединяются, после чего классификатор предсказывает вероятность класса.

## Почему выбран именно этот подход
Хотелось проверить гипотезу, что для детекции synthetic/fake-лиц полезны не только обычные RGB-признаки, но и артефакты, выделяемые через residual-представления.

## Обучение
- функция потерь: BCEWithLogitsLoss с pos_weight
- основная метрика: F1-score
- после обучения дополнительно подбирался threshold для улучшения F1
- для финального решения использовался threshold = 0.45

## Итог
Финальная версия решения - DualBranchResidualNet с multi-channel residual-входом и порогом 0.45.